In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os.path as p

%matplotlib ipympl

In [ ]:
from pathlib import Path

data_folder = Path(r"E:\output\nad10\c01")

event_file = data_folder / "nad10-c01-ai4.evt.cat"

In [ ]:
# read event file
event_file = p.join(data_folder, 'nad10-c01-ai4.evt.cat')
# format of csv file is with times in ms and event type:
#     215180 Frame
#     215212 Frame
#     215245 Frame
events = pd.read_csv(
    event_file,
    sep=r"\s+",
    header=None,
    names=['time', 'type']
)
frame_onset_ms = events['time'].values

len(frame_onset_ms)


In [ ]:
# read data file with 16-bit resolution
data_file = p.join(data_folder, 'nad10-c01.dat')
data = np.fromfile(data_file, dtype=np.int16)

data = np.reshape( data, (-1, 32) )
data.shape


In [ ]:
channel = 12
fs = 40000  # Hz

fig, ax = plt.subplots(figsize=(10,4))
ax.set_facecolor("white")

t_sec = np.arange(data.shape[0]) / fs
ax.plot(t_sec, data[:, channel], color='black', linewidth=0.6, zorder=2)

# convert ms -> s
frame_onset_sec = frame_onset_ms / 1000

# Detect the 3 distinct blocks via the large gaps
gaps = np.diff(frame_onset_sec)
break_indices = np.where(gaps > 1)[0]
starts = np.concatenate([[0], break_indices + 1])
ends = np.concatenate([break_indices, [len(frame_onset_sec) - 1]])

for s, e in zip(starts, ends):
    ax.axvspan(frame_onset_sec[s], frame_onset_sec[e], color="#5e915e", alpha=0.5, zorder=0)

ax.set_xlabel('Time (s)')
ax.set_ylabel(f'Channel {channel} (a.u.)')
ax.set_title('Raw signal with frame blocks')

plt.tight_layout()
plt.savefig('channel12_frame_blocks.svg', format='svg', dpi=300)
plt.show()

In [ ]:
# convert to sample indices (int)
frame_index = (frame_onset_ms * 40).astype(int)   # 40 kHz

start_index = 935180 *40
end_index   = 1258369.600 *40

# select even frames within the window
sel_frames = frame_index[::2]
sel_frames = sel_frames[(sel_frames >= start_index) & (sel_frames <= end_index)]

pre_frames  = 200
post_frames = 2800

if len(sel_frames) == 0:
    print("No frame in the selected range.")
else:
    # extract the segments
    surrounding_data = np.zeros((len(sel_frames), pre_frames+post_frames))
    for i, f in enumerate(sel_frames):
        surrounding_data[i, :] = data[f-pre_frames:f+post_frames, channel]

    # time vector in ms
    t = (np.arange(pre_frames + post_frames) - pre_frames) * 1000 / 40000

    plt.figure()

    # 10 random traces in light gray
    n_plot = min(10, len(sel_frames))
    for i in np.random.randint(0, len(sel_frames), size=n_plot):
        plt.plot(t, surrounding_data[i, :], color='lightgray', alpha=0.7, lw=1)
    # mean trace in thick blue
    mean_trace = np.mean(surrounding_data, axis=0)
    plt.plot(t, mean_trace, color='blue', lw=3)

    # standard deviation in thin black
    plt.plot(t, np.std(surrounding_data, axis=0), lw=1, color='black')

    # vertical reference lines in ms
    plt.axvline(0, color='red')                                # onset
    plt.axvline(1292 * 1000 / 40000, color='green')            # offset
    plt.axvline(2*1292 * 1000 / 40000, color='red')             # next onset

    plt.xlabel("Time (ms)")
    plt.ylabel("Amplitude (int16)")
    out_pdf = p.join(data_folder, f"peri_frame_ch{channel}.pdf")
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight")  # save BEFORE plt.show()
    print("PDF saved:", out_pdf)
    plt.show()

In [ ]:
# === create the template to remove the artifact ===
template = np.zeros(data.shape[0])

# mean template (from alignment on the onsets)
aligned_template = np.mean(surrounding_data, axis=0)[pre_frames:]

even_frames = frame_index[::2].astype(int)

for i, f in enumerate(even_frames[:-1]):
    next_f = even_frames[i+1]
    len_to_next = next_f - f
    
    # available length in the segment
    L = min(len_to_next, len(aligned_template))
    
    # fill only what fits
    template[f:f+L] = aligned_template[:L]
    

In [ ]:
plt.figure()
plt.plot(template, lw=1)
plt.plot(data[:,channel], lw=0.5, alpha=0.5)

plt.plot(data[:,channel] - template - 2000)

### Repeat the same procedure for all channels

In [ ]:
data.shape

In [ ]:
nr_channels = data.shape[1]

# === sample indices (40 kHz) ===
frame_index = (frame_onset_ms * 40).astype(int)

start_index = 935180 *40
end_index   = 1258369.600 *40

# select even frames within the window
sel_frames = frame_index[::2]
sel_frames = sel_frames[(sel_frames >= start_index) & (sel_frames <= end_index)]

# windows in samples (adapted for 40 kHz)
pre_frames  = 200    # 5 ms
post_frames = 2800   # 70 ms

# extraction (#frames, #samples, #channels)
surrounding_data = np.zeros((len(sel_frames), pre_frames+post_frames, nr_channels))
for i, f in enumerate(sel_frames):
    surrounding_data[i, :, :] = data[f-pre_frames:f+post_frames, :]

print("surrounding_data shape:", surrounding_data.shape)
    

In [ ]:
# === build the global template ===
template = np.zeros_like(data)

# mean template aligned on the onsets (shape: #samples, #channels)
aligned_template = np.mean(surrounding_data, axis=0)[pre_frames:, :]

even_frames = frame_index[::2].astype(int)

for i, f in enumerate(even_frames[:-1]):
    next_f = even_frames[i+1]
    len_to_next = next_f - f
    
    # usable length = minimum of the interval and the template size
    L = min(len_to_next, aligned_template.shape[0])
    
    # fill channel by channel
    template[f:f+L, :] = aligned_template[:L, :]

# === correction ===
corrected_data = data - template


In [ ]:
sampling_rate = 40000

# mean artifact
mean_artifact = np.mean(surrounding_data, axis=0)

# time axis based on the true temporal dimension
n_time = mean_artifact.shape[0]
t = (np.arange(n_time) - pre_frames) * 1000 / sampling_rate

fig, axes = plt.subplots(1,2,figsize=(7,4))

# subplot 1
for i, ch in enumerate(range(nr_channels-1, -1, -1)):
    axes[0].plot(t, mean_artifact[:,ch] + i*100, lw=1)

axes[0].set_yticks([i*100 for i in range(nr_channels)])
axes[0].set_yticklabels(range(nr_channels-1,-1,-1))

axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Channel")
axes[0].tick_params(axis='y', labelsize=8)

# just zoom the visible window
axes[0].set_xlim(-10, 75)


# artifact amplitude
amplitudes = np.max(mean_artifact, axis=0) - np.min(mean_artifact, axis=0)

axes[1].bar(np.arange(nr_channels), amplitudes)
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Artifact amplitude (a.u.)")

fig.tight_layout()

out_pdf = p.join(data_folder, "artifact_summary.pdf")
fig.savefig(out_pdf, format="pdf", bbox_inches="tight")

plt.show()
print(pre_frames)
print(surrounding_data.shape)

In [ ]:
# plot overlay of original and corrected data for a time window
time_window = slice(37407200, 37412200)
plt.figure()

channel = 12
plt.plot(data[time_window, channel], lw=1, alpha=0.5, label='original')
plt.plot(corrected_data[time_window, channel], lw=1, alpha=0.5, label='corrected')
plt.plot(template[time_window, channel], lw=1, alpha=0.5, label='template')
plt.legend()

In [ ]:
sampling_rate = 40000

# mean artifact
mean_artifact = np.mean(surrounding_data, axis=0)

# time axis based on the true temporal dimension
n_time = mean_artifact.shape[0]
t = (np.arange(n_time) - pre_frames) * 1000 / sampling_rate

fig, axes = plt.subplots(1,2,figsize=(7,4))

# subplot 1
for i, ch in enumerate(range(nr_channels-1, -1, -1)):
    axes[0].plot(t, mean_artifact[:,ch] + i*100, lw=1)

axes[0].set_yticks([i*100 for i in range(nr_channels)])
axes[0].set_yticklabels(range(nr_channels-1,-1,-1))

axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Channel")
axes[0].tick_params(axis='y', labelsize=8)

# just zoom the visible window
axes[0].set_xlim(-10, 75)
axes[0].margins(x=0.02)

# artifact amplitude
amplitudes = np.max(mean_artifact, axis=0) - np.min(mean_artifact, axis=0)

axes[1].bar(np.arange(nr_channels), amplitudes)
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Artifact amplitude (a.u.)")

fig.tight_layout()

out_pdf = p.join(data_folder, "artifact_summary.pdf")
fig.savefig(out_pdf, format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
from scipy.signal import welch

s1 = slice(30007200, 34000000)
s2 = slice(48412376, 52412376)

fs = 40000
nperseg = 80000

# PSD original
f_orig_1, Pxx_orig_1 = welch(data[s1, channel], fs=fs, nperseg=nperseg)
f_orig_2, Pxx_orig_2 = welch(data[s2, channel], fs=fs, nperseg=nperseg)

# PSD corrected
f_corr_2, Pxx_corr_2 = welch(corrected_data[s2, channel], fs=fs, nperseg=nperseg)

fig, axes = plt.subplots(1, 2, figsize=(10,4))

# --- full PSD ---
axes[0].semilogy(f_orig_1, Pxx_orig_1, label='original segment 1')
axes[0].semilogy(f_orig_2, Pxx_orig_2, label='original segment 2')
axes[0].semilogy(f_corr_2, Pxx_corr_2, label='corrected segment 2')

axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power spectral density (a.u.)')
axes[0].set_xlim(0, 10000)
axes[0].legend()

# --- low-frequency zoom ---
axes[1].semilogy(f_orig_1, Pxx_orig_1)
axes[1].semilogy(f_orig_2, Pxx_orig_2)
axes[1].semilogy(f_corr_2, Pxx_corr_2)

axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_xlim(0, 1000)
axes[1].set_title('Zoom 0–1000 Hz')

fig.tight_layout()

out_pdf = p.join(data_folder, f"PSD_comparison_ch{channel}.pdf")
fig.savefig(out_pdf, format="pdf", bbox_inches="tight")

print("Figure saved:", out_pdf)

plt.show()

In [ ]:
## filter data with a 4-th order Butterworth bandpass filter between 300 and 5000 Hz

filt_data = corrected_data.copy()
from scipy.signal import butter, filtfilt
b, a = butter(4, [300/(20000/2), 5000/(20000/2)], btype='bandpass')
filt_data = filtfilt(b, a, filt_data, axis=0)
filt_data.shape

In [ ]:
## plot power spectrum of filtered data for selected channel

from scipy.signal import welch

s1 = slice(30007200, 34000000)
s2 = slice(48412376, 52412376)

f_filt_2, Pxx_filt_2 = welch(filt_data[s2,channel], fs=40000, nperseg=40000)

plt.figure()
plt.semilogy(f_corr_2, Pxx_corr_2, label='corrected 6-8M')
plt.semilogy(f_filt_2, Pxx_filt_2, label='filtered 6-8M')
plt.xlabel('Frequency (Hz)')    
plt.ylabel('Power spectral density (a.u.)')

plt.legend()



In [ ]:
ex_frame = 6178000

In [ ]:
1/ (1 / 1292 * 40)

In [ ]:
np.mean( np.diff( sel_frames ) ) / 2

In [ ]:
np.mean( np.diff( frame_index ) )

In [ ]:
# === Corrected data ===
corrected_data = data - template    

# === Save as .dat int16 (Neuroscope) ===
out_file = r"E:\output\nad10\c01\corrected\nad10-c01.dat"  # raw string : the leading r avoids problems with \n, \t, etc.

out_data = corrected_data.astype('<i2')  # int16 little-endian
out_data.tofile(out_file)

print(f"File saved: {out_file}")